# WESAD Wrist EDA

The purpose for this notebook is to understand the wrist-only portion of the WESAD dataset that we will use to train the MindWave stress-detection model.

For this project, we extracted only the **Empatica E4** (equivalent to a smartwatch) signals:

| Signal | Sampling rate | Description |
|--------|--------------:|-------------|
| BVP    | 64 Hz         | Blood Volume Pulse (PPG) → source of HRV |
| EDA    | 4 Hz          | Electrodermal Activity (cEDA proxy) |
| TEMP   | 4 Hz          | Skin temperature |
| ACC    | 32 Hz, 3-axis | Accelerometer |

WESAD label scheme: `0=undefined, 1=baseline, 2=stress, 3=amusement, 4=meditation, 5/6/7=ignore`. We will keep only `{1, 2}`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch
from collections import Counter

from src.config import WRIST_FS, LABEL_FS, LABEL_NAMES
from src.wesad_loader import load_subject, iter_subjects, expected_lengths
from src.preprocessing import acc_magnitude

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

### Inspect a single subject

In [ ]:
rec = load_subject('S2')
lengths = expected_lengths(rec)
fs_table = pd.DataFrame({
    'samples': lengths,
    'fs (Hz)': {**WRIST_FS, 'label_700Hz': LABEL_FS},
})
fs_table['duration (s)'] = (fs_table['samples'] / fs_table['fs (Hz)']).round(1)
fs_table

### Plot data for a full session for the selected subject, using all 4 wrist signals.

In [ ]:
WESAD_LABEL_NAMES = {0: 'undef', 1: 'baseline', 2: 'stress', 3: 'amusement',
                     4: 'meditation', 5: 'ign5', 6: 'ign6', 7: 'ign7'}
WESAD_LABEL_COLOURS = {1: '#9ad3bc', 2: '#f76c5e', 3: '#ffd166',
                       4: '#a0c4ff', 0: '#eeeeee', 5: '#dddddd',
                       6: '#dddddd', 7: '#dddddd'}

def label_segments(label_700hz):
    """Yield (start_s, end_s, label) tuples for contiguous label runs."""
    diffs = np.diff(label_700hz)
    boundaries = np.r_[0, np.where(diffs != 0)[0] + 1, len(label_700hz)]
    for a, b in zip(boundaries[:-1], boundaries[1:]):
        yield a / LABEL_FS, b / LABEL_FS, int(label_700hz[a])

def shade_labels(ax, label_700hz):
    seen = set()
    for s, e, lab in label_segments(label_700hz):
        ax.axvspan(s, e, color=WESAD_LABEL_COLOURS.get(lab, '#dddddd'),
                   alpha=0.35,
                   label=WESAD_LABEL_NAMES.get(lab, str(lab)) if lab not in seen else None)
        seen.add(lab)

fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
signals = [
    ('BVP',  rec.bvp,                       WRIST_FS['BVP']),
    ('EDA',  rec.eda,                       WRIST_FS['EDA']),
    ('TEMP', rec.temp,                      WRIST_FS['TEMP']),
    ('ACC mag', acc_magnitude(rec.acc),     WRIST_FS['ACC']),
]
for ax, (name, sig, fs) in zip(axes, signals):
    t = np.arange(len(sig)) / fs
    shade_labels(ax, rec.label_700hz)
    ax.plot(t, sig, lw=0.5, color='#222')
    ax.set_ylabel(name)
axes[-1].set_xlabel('time (s)')
axes[0].legend(loc='upper right', ncol=5, fontsize=8)
fig.suptitle(f'WESAD wrist signals — subject {rec.subject}', y=0.995)
fig.tight_layout()

### 60 s windows for the selected labels, normal vs stress


In [ ]:
from src.config import LABEL_REMAP

def first_window_for_label(label_700hz, target_label, win_s=60):
    target_count = int(win_s * LABEL_FS)
    for start in range(0, len(label_700hz) - target_count, LABEL_FS):
        seg = label_700hz[start:start + target_count]
        if (seg == target_label).mean() > 0.95:
            return start / LABEL_FS
    return None

fig, axes = plt.subplots(2, 4, figsize=(15, 5), sharey='col')
for row, lab in enumerate([1, 2]):   # 1=baseline → normal, 2=stress
    t0 = first_window_for_label(rec.label_700hz, lab)
    if t0 is None:
        continue
    for col, (name, sig, fs) in enumerate(signals):
        a, b = int(t0 * fs), int((t0 + 60) * fs)
        seg = sig[a:b]
        axes[row, col].plot(np.arange(len(seg)) / fs, seg, lw=0.6)
        if row == 0:
            axes[row, col].set_title(name)
        if col == 0:
            axes[row, col].set_ylabel(LABEL_NAMES[LABEL_REMAP[lab]])
axes[-1, 0].set_xlabel('time (s)')
fig.suptitle('60 s zoom — class-conditioned signals', y=1.0)
fig.tight_layout()


### Label distribution

In [ ]:
counts = Counter(rec.label_700hz.tolist())
df = pd.DataFrame({
    'label': [WESAD_LABEL_NAMES.get(k, str(k)) for k in counts],
    'samples_700Hz': list(counts.values()),
}).sort_values('samples_700Hz', ascending=False)
df['seconds'] = df['samples_700Hz'] / LABEL_FS
fig, ax = plt.subplots(figsize=(8, 3.5))
sns.barplot(df, x='label', y='seconds', ax=ax,
            palette=[WESAD_LABEL_COLOURS.get(k, '#888') for k in counts])
ax.set_title(f'Label durations (subject {rec.subject})')
ax.set_ylabel('duration (s)')
for c in ax.containers:
    ax.bar_label(c, fmt='%.0f s', padding=2)

## Class-balance heatmap across all subjects

In [ ]:
rows = []
for r in iter_subjects():
    c = Counter(r.label_700hz.tolist())
    row = {'subject': r.subject}
    for lab in [1, 2, 3, 4]:
        row[WESAD_LABEL_NAMES[lab]] = c.get(lab, 0) / LABEL_FS
    rows.append(row)
subj_df = pd.DataFrame(rows).set_index('subject')
subj_df['total_kept_min'] = subj_df[['baseline', 'stress', 'amusement']].sum(axis=1) / 60
display(subj_df.round(1))

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(subj_df.drop(columns='total_kept_min'), annot=True, fmt='.0f',
            cmap='viridis', cbar_kws={'label': 'seconds per class'}, ax=ax)
ax.set_title('Per-subject class duration (s)')

## BVP power spectral density per class
Sanity check that cardiac content (1–2 Hz) is present and that filter cut-offs are appropriate

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for lab in [1, 2]:   # 1=baseline → normal, 2=stress
    t0 = first_window_for_label(rec.label_700hz, lab, win_s=60)
    if t0 is None: continue
    fs = WRIST_FS['BVP']
    seg = rec.bvp[int(t0 * fs):int((t0 + 60) * fs)]
    f, Pxx = welch(seg, fs=fs, nperseg=min(512, len(seg)))
    ax.semilogy(f, Pxx, label=LABEL_NAMES[LABEL_REMAP[lab]])
ax.set_xlim(0, 10)
ax.set_xlabel('frequency (Hz)'); ax.set_ylabel('PSD')
ax.set_title('BVP PSD per class (subject S2)')
ax.axvspan(0.5, 8, color='#bbb', alpha=0.2, label='band-pass')
ax.legend()